In [1]:
# Centralized imports (cleaned)
from bioio import BioImage
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tifffile import imwrite, imread
import tifffile
from skimage.segmentation import expand_labels, clear_border
from skimage.measure import regionprops_table
from cellpose import models
import napari
from liffile import LifFile
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import colorsys
from scipy.spatial import KDTree
from skimage.measure import regionprops
from skimage.transform import resize, rescale
from pathlib import Path
import pandas as pd
import scipy.ndimage as ndi
from skimage.measure import regionprops
from instanseg import InstanSeg
import time
from cellpose import models, utils as cellpose_utils
# from micro_sam.automatic_segmentation import get_predictor_and_segmenter, automatic_3d_segmentation

available = torch.cuda.is_available()
device_count = torch.cuda.device_count() if available else 0
device_name = torch.cuda.get_device_name(0) if available and device_count > 0 else None

status = {
    "cuda_available": available,
    "device_count": device_count,
    "device_name": device_name,
}
print(status)

{'cuda_available': True, 'device_count': 1, 'device_name': 'NVIDIA GeForce RTX 5080'}


In [2]:

_MODEL_CACHE = {}  # cache loaded models across instances so we don't reload them per image

class SegmentationComparisons:
    """Compare three nuclear-segmentation methods on the same DAPI z-stack.

    The input image is a 2-channel z-stack stored as (Z, C, Y, X); only the
    DAPI channel (`dapi_channel`) is used. The three methods are:
      - `original`:        custom cellpose model, per-z 2D then stitched in 3D.
                           This is the BASELINE the other two are compared against.
      - `cellpose_true3d`: custom cellpose model, native 3D.
      - `instanseg`:       InstanSeg per-z (nuclei) then stitched in 3D.

    Per image we save one TIF per method plus a single 6-panel comparison PNG.
    """

    def __init__(self, input_csv, index, scale_factor_xy=3, scale_factor_z=2,
        custom_model_path=r"Z:\Bel\Jorge_SPACEFISH_Examples\v115-2ch\spacefish_custom_tissue",
        output_dir=Path(r"Z:\Bel\Jorge_SPACEFISH_Examples\cellpose_comparisons\round_2_comparisons"),
        dapi_channel=0):
        self.input_csv = input_csv
        self.index = index
        self.scale_factor_xy = float(scale_factor_xy)
        self.scale_factor_z = float(scale_factor_z)
        self.custom_model_path = custom_model_path

        row = input_csv.loc[index]
        self.two_channel_image_path = Path(row["2_channel_tif_save_path"])
        self.dapi_channel = int(dapi_channel)
        self.image_name = row["image_name"]

        self.output_dir = Path(output_dir)

        # (Z, C, Y, X)
        self.two_channel_image = tifffile.imread(self.two_channel_image_path)
        if self.two_channel_image.ndim != 4:
            raise ValueError(f"Expected a 4D (Z, C, Y, X) image, got shape {self.two_channel_image.shape}")

        self.results = {}  # method -> label array
        self.counts = {}   # method -> object count

    def pixel_size(self):
        with tifffile.TiffFile(self.two_channel_image_path) as tif:
            tags = {tag.name: tag.value for tag in tif.pages[0].tags.values()}
            x_um = 1 / (tags["XResolution"][0] / tags["XResolution"][1])
            y_um = 1 / (tags["YResolution"][0] / tags["YResolution"][1])
            try:
                z_um = float(str(tags["IJMetadata"]).split("nscales=")[1].split(",")[2].split("\\nunit")[0])
            except Exception:
                z_um = float(str(tags["ImageDescription"]).split("spacing=")[1].split("loop")[0])
        self.original_spacing = (x_um, y_um, z_um)

    def rescale_image(self):
        """Downsample the DAPI channel and compute z-anisotropy."""
        self.pixel_size()
        x_um, y_um, z_um = self.original_spacing
        xy_ratio = 1.0 / self.scale_factor_xy
        z_ratio = 1.0 / self.scale_factor_z

        dapi = self.two_channel_image[:, self.dapi_channel].astype(np.float32)  # (Z, Y, X)
        self.dapi_ds = rescale(dapi, (z_ratio, xy_ratio, xy_ratio),
                               anti_aliasing=True, preserve_range=True).astype(np.float32)

        # physical voxel spacing after downsampling
        self.z_spacing_ds = z_um * self.scale_factor_z
        self.xy_spacing_ds = x_um * self.scale_factor_xy
        self.anisotropy = self.z_spacing_ds / self.xy_spacing_ds
        self.pixel_size_um_2d = float(self.xy_spacing_ds)
        print(f"downsampled DAPI shape {self.dapi_ds.shape}, anisotropy {self.anisotropy:.3f}")

    @staticmethod
    def _get_cellpose_model(pretrained=None):
        key = str(pretrained)
        if key not in _MODEL_CACHE:
            if pretrained is None:
                _MODEL_CACHE[key] = models.CellposeModel(gpu=True)  # built-in CPSAM
            else:
                _MODEL_CACHE[key] = models.CellposeModel(gpu=True, pretrained_model=str(pretrained))
        return _MODEL_CACHE[key]

    # ---- segmentation methods: each RETURNS a label volume ----

    def cellpose_custom_2d_stitched(self, stitch_threshold=0.1, cellprob_threshold=0.0, min_size=15):
        """BASELINE 'original': custom cellpose model on DAPI, per-z 2D then stitched in 3D."""
        model = self._get_cellpose_model(self.custom_model_path)
        seg, _, _ = model.eval(self.dapi_ds, z_axis=0, do_3D=False,
                               stitch_threshold=stitch_threshold, anisotropy=self.anisotropy,
                               cellprob_threshold=cellprob_threshold, min_size=min_size)
        return seg

    def cellpose_custom_3d(self, cellprob_threshold=0.0, min_size=15):
        """Custom cellpose model on DAPI, native 3D (stitch_threshold is unused in do_3D mode)."""
        model = self._get_cellpose_model(self.custom_model_path)
        seg, _, _ = model.eval(self.dapi_ds, z_axis=0, do_3D=True, anisotropy=self.anisotropy,
                               cellprob_threshold=cellprob_threshold, min_size=min_size)
        return seg

    def instanseg_2d_stitched(self, stitch_threshold=0.1, pixel_size_scale=1.0, target="nuclei"):
        """InstanSeg per-z (DAPI only) then stitched in 3D.

        - `stitch_threshold`: IoU cutoff for linking 2D masks across z.
        - `pixel_size_scale`: multiplies the pixel size handed to InstanSeg (object-scale prior).
        - `target`: which InstanSeg output to keep ('nuclei' or 'cells').
        """
        if "instanseg" not in _MODEL_CACHE:
            _MODEL_CACHE["instanseg"] = InstanSeg("fluorescence_nuclei_and_cells", verbosity=0)
        model = _MODEL_CACHE["instanseg"]
        pixel_size = self.pixel_size_um_2d * float(pixel_size_scale)
        z_masks = []
        for z in range(self.dapi_ds.shape[0]):
            plane = self.dapi_ds[z][None]  # (C=1, H, W), DAPI only
            labeled, _ = model.eval_small_image(plane, pixel_size, target=target)
            lab = np.asarray(labeled.cpu() if hasattr(labeled, "cpu") else labeled).squeeze()
            if lab.ndim == 3:  # (n_outputs, H, W) -> take the first output
                lab = lab[0]
            z_masks.append(lab.astype(np.uint32))
        stacked = np.stack(z_masks, axis=0)
        return cellpose_utils.stitch3D(stacked, stitch_threshold=stitch_threshold)

    # ---- output helpers ----

    @staticmethod
    def _align(arr, ref_shape):
        """Nearest-neighbour resize to ref_shape if shapes differ (handles +/-1px rounding)."""
        if arr.shape == ref_shape:
            return arr
        return resize(arr, ref_shape, order=0, anti_aliasing=False,
                      preserve_range=True).astype(arr.dtype)

    def _save_comparison_png(self, mips, counts, png_path):
        """Single 6-panel figure (2x3):
          row 0: original image (DAPI) | cellpose true 3D overlay  | instanseg overlay
          row 1: original seg overlay  | diff cellpose 3D vs orig   | diff instanseg vs orig
        Diff panels: green = method has it but original doesn't; red = original has it but method doesn't.
        """
        dapi_mip = mips["dapi"]
        original = mips["original"]
        cp3d = mips["cellpose_true3d"]
        insta = mips["instanseg"]

        def overlay(ax, labels, title):
            ax.imshow(dapi_mip, cmap="gray")
            if labels is not None:
                masked = np.ma.masked_where(labels == 0, labels)
                ax.imshow(masked, cmap="nipy_spectral", alpha=0.5, interpolation="nearest")
            ax.set_title(title)

        base_norm = dapi_mip.astype(np.float32)
        base_norm = (base_norm - base_norm.min()) / (base_norm.max() - base_norm.min() + 1e-8)

        def diff(ax, other, title):
            rgb = np.stack([base_norm, base_norm, base_norm], axis=-1)
            if original is not None and other is not None:
                orig_b = original > 0
                other_b = other > 0
                rgb[other_b & ~orig_b] = [0.0, 1.0, 0.0]  # green: in method but not original
                rgb[orig_b & ~other_b] = [1.0, 0.0, 0.0]  # red: in original but not method
            ax.imshow(rgb, interpolation="nearest")
            ax.set_title(title)

        fig, axes = plt.subplots(2, 3, figsize=(21, 12))
        # Row 0
        axes[0, 0].imshow(dapi_mip, cmap="gray")
        axes[0, 0].set_title("Original image (DAPI max projection)")
        overlay(axes[0, 1], cp3d,    f"Cellpose true 3D  n={counts.get('cellpose_true3d', '?')}")
        overlay(axes[0, 2], insta,   f"InstanSeg  n={counts.get('instanseg', '?')}")
        # Row 1
        overlay(axes[1, 0], original, f"Original segmentation (cellpose 2D stitched)  n={counts.get('original', '?')}")
        diff(axes[1, 1], cp3d,  "Diff: cellpose 3D vs original\n(green=3D only, red=original only)")
        diff(axes[1, 2], insta, "Diff: InstanSeg vs original\n(green=InstanSeg only, red=original only)")

        for ax in axes.ravel():
            ax.axis("off")
        fig.suptitle(self.image_name)
        fig.tight_layout()
        fig.savefig(png_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

    ######### MAIN PART ############
    def run(self,
            cellpose_stitch_threshold=0.1,
            cellpose_min_size=15,
            cellprob_threshold=0.0,
            instanseg_stitch_threshold=0.1,
            instanseg_pixel_size_scale=1.0,
            instanseg_target="nuclei"):
        """Run the three methods, save one TIF per method + one 6-panel PNG, and
        return one results row per successful method. All stitch thresholds (and the
        3D cellprob threshold) are tunable here; defaults match the pipeline config."""
        self.rescale_image()
        self.output_dir.mkdir(parents=True, exist_ok=True)

        methods = [
            ("original", lambda: self.cellpose_custom_2d_stitched(
                stitch_threshold=cellpose_stitch_threshold,
                cellprob_threshold=cellprob_threshold, min_size=cellpose_min_size)),
            ("cellpose_true3d", lambda: self.cellpose_custom_3d(
                cellprob_threshold=cellprob_threshold, min_size=cellpose_min_size)),
            ("instanseg", lambda: self.instanseg_2d_stitched(
                stitch_threshold=instanseg_stitch_threshold,
                pixel_size_scale=instanseg_pixel_size_scale, target=instanseg_target)),
        ]

        mips = {"dapi": self.dapi_ds.max(axis=0)}
        ref_shape = mips["dapi"].shape
        run_results = []
        for name, fn in methods:
            try:
                t0 = time.time()
                seg = fn()
                time_taken = time.time() - t0

                count = int(np.unique(seg).size - (1 if (seg == 0).any() else 0))
                self.results[name] = seg
                self.counts[name] = count
                mips[name] = self._align(seg.max(axis=0) if seg.ndim == 3 else seg, ref_shape)
                print(f"  {name}: {count} objects, {time_taken:.1f}s")

                imwrite(self.output_dir / f"{self.image_name}__{name}.tif", seg.astype(np.uint32))
                run_results.append({"image_name": self.image_name, "method": name,
                                    "time_taken": time_taken, "objects_found": count})
            except Exception as exc:
                mips[name] = None
                print(f"[SKIP] {name}: {type(exc).__name__}: {exc}")

        # ensure all expected keys exist for the figure even if a method was skipped
        for name in ("original", "cellpose_true3d", "instanseg"):
            mips.setdefault(name, None)

        self._save_comparison_png(mips, self.counts, self.output_dir / f"{self.image_name}__comparison.png")
        return run_results


In [3]:
input_csv = pd.read_excel(r"z:\Bel\Jorge_SPACEFISH_Examples\image_locations.xlsx")
input_csv.head()


,image_name,path,dapi_channel,bf_channel,image_type,scene_id,"censor region (z1,z2,x1_z1, x2_z1, y1_z1,y2_z1,x1_z2,x2_z2,y1_z2,y2_z2)",existing_segmentation_path,2_channel_tif_save_path
0,dev4_1_6h,Z:\Jorge\20241124_6h_dev4\20241124_dev4_6h_mer...,6,7,vascu,0,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
1,dev4_2_6h,Z:\Jorge\20241125_repeats_6h_2d_pin255\2024112...,6,7,vascu,0,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
2,dev4_3_6h,Z:\Jorge\20241124_6h_dev4\20241124_dev4_6h_mer...,6,7,vascu,2,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
3,dev7_3_day1,Z:\Jorge\20241126_day1_dev7\20241126_dev7_day1...,6,7,vascu,0,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
4,dev7_2_day1,Z:\Jorge\20241126_day1_dev7\20241126_dev7_day1...,6,7,vascu,1,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...


In [4]:
# run the three-method comparison on every image
# reversed() so images at the end of the list (not yet processed) run first on restart
all_results = []
for index in (input_csv.index):
    comparison = SegmentationComparisons(input_csv, index=index, scale_factor_xy=3, scale_factor_z=2)
    image_results = comparison.run()
    all_results.extend(image_results)

    # write this image's results immediately
    image_df = pd.DataFrame(image_results)
    safe_name = "".join(c if c.isalnum() or c in "-_." else "_" for c in str(comparison.image_name))
    image_csv_path = comparison.output_dir / f"results_{safe_name}.csv"
    image_df.to_csv(image_csv_path, index=False)
    print(f"saved {len(image_df)} rows -> {image_csv_path}")

# single combined CSV across all images and methods
results_df = pd.DataFrame(all_results)
results_csv_path = comparison.output_dir / "segmentation_comparison_results.csv"
results_df.to_csv(results_csv_path, index=False)
print(f"saved {len(results_df)} rows -> {results_csv_path}")
results_df

# to tune thresholds later, pass them to run(), e.g.:
# comparison.run(cellpose_stitch_threshold=0.2, instanseg_stitch_threshold=0.1, cellprob_threshold=0.0)


downsampled DAPI shape (63, 1302, 1311), anisotropy 2.838


c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\cellpose\dynamics.py:524: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),
no seeds found in get_masks_torch - no masks found.
100%|██████████| 62/62 [00:04<00:00, 14.10it/s]


  original: 1780 objects, 173.3s
  cellpose_true3d: 1728 objects, 793.3s


c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
 16%|█▌        | 10/62 [00:00<00:03, 14.33it/s]c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\cellpose\metrics.py:176: RuntimeWarning: invalid value encountered in divide
  iou = overlap / (n_pixels_pred + n_pixels_true - overlap)
100%|██████████| 62/62 [00:04<00:00, 13.54it/s]


  instanseg: 1926 objects, 20.8s
saved 3 rows -> Z:\Bel\Jorge_SPACEFISH_Examples\cellpose_comparisons\round_2_comparisons\results_dev4_1_6h.csv
downsampled DAPI shape (63, 1304, 1298), anisotropy 2.816


...

100%|██████████| 47/47 [00:00<00:00, 62.86it/s]


  original: 70 objects, 24.5s
  cellpose_true3d: 63 objects, 179.0s


100%|██████████| 47/47 [00:01<00:00, 44.98it/s]


  instanseg: 70 objects, 3.8s
saved 3 rows -> Z:\Bel\Jorge_SPACEFISH_Examples\cellpose_comparisons\round_2_comparisons\results_dev1_2_P1.csv
saved 120 rows -> Z:\Bel\Jorge_SPACEFISH_Examples\cellpose_comparisons\round_2_comparisons\segmentation_comparison_results.csv


,image_name,method,time_taken,objects_found
0,dev4_1_6h,original,173.324524,1780
1,dev4_1_6h,cellpose_true3d,793.295342,1728
2,dev4_1_6h,instanseg,20.811819,1926
3,dev4_2_6h,original,137.936471,1473
4,dev4_2_6h,cellpose_true3d,650.662447,1410
...,...,...,...,...
115,dev1_2_R_6,cellpose_true3d,642.863443,259
116,dev1_2_R_6,instanseg,15.644640,292
117,dev1_2_P1,original,24.472483,70
118,dev1_2_P1,cellpose_true3d,179.043669,63
